# Сетевой анализ нейронов и валидация на данных Anton-Sanchez et al.

Этот ноутбук работает с каноническим форматом линейной 3D-сети: `vertices.csv`, `edges.csv`, `spines.csv`, `matrix.npy`, `metadata.json`. Такой формат напрямую соответствует данным Anton-Sanchez et al. (`vertices`, `m`, `X`) и используется как промежуточный стандарт для MICrONS после предобработки.

Для данных статьи `metadata.json` не содержит координату сомы. В статье используется корень дерева `r`, а не mesh сомы; поэтому по умолчанию используется `root_vertex_id = 0`. Для MICrONS реальная координата сомы сохраняется отдельно в `metadata.json` как `soma_point`, а `root_vertex_id` задаёт вершину сети, от которой считаются shortest-path distances.

In [ ]:
from pathlib import Path
import hashlib
import importlib
import itertools
import json
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

network_module = importlib.import_module("dendrite_analysis.network")
network_module = importlib.reload(network_module)

compute_simulation_envelopes = network_module.compute_simulation_envelopes
estimate_smooth_intensity = network_module.estimate_smooth_intensity
fit_inhomogeneous_poisson = network_module.fit_inhomogeneous_poisson
load_standard_network = network_module.load_standard_network
network_circumradius = network_module.network_circumradius
project_spines_to_graph = network_module.project_spines_to_graph
ripley_k_network = network_module.ripley_k_network
run_analysis = network_module.run_analysis

## Конфигурация

Для прямого воспроизведения статьи используем `N_SIMULATIONS = 19`, global constant-width envelope, geometrically corrected inhomogeneous network K-function и log-quadratic intensity по расстоянию до корня. Для более стабильных рабочих прогонов можно увеличить `N_SIMULATIONS` до 99.

In [ ]:
DATA_ROOT = Path("3Dnetworks_export")
OUTPUT_DIR = Path("output_anton_sanchez_validation")
ANALYSIS_CACHE_DIR = OUTPUT_DIR / "_analysis_cache"

ROOT_VERTEX_ID = 0
COUNT_SOMA_AS_BRANCH_POINT = True
N_SIMULATIONS = 19
N_JOBS = 8
N_R_VALUES = 20
N_PERMUTATIONS = 1000
CIRCUMRADIUS_MARGIN = 0.02
FIG6_COMMON_GRID_N = 300
FIG6C_R_MAX = 165.96
INTENSITY_ENVELOPE_BOOTSTRAPS = 200
INTENSITY_ENVELOPE_ALPHA = 0.05
RANDOM_STATE = 42

# Быстрые тесты: поставь NETWORK_LIMIT=1 и RUN_FULL_ANALYSIS=True.
NETWORK_LIMIT = None
EXCLUDED_BASAL_DENDRITES = "basal_08"
RUN_FULL_ANALYSIS = False
RUN_GROUP_COMPARISON = False
USE_ANALYSIS_CACHE = True
FORCE_RECOMPUTE_ANALYSIS = False

def normalize_excluded_dendrites(excluded):
    if excluded is None or excluded == "":
        return set()
    if isinstance(excluded, str):
        return {excluded}
    return {str(name) for name in excluded if str(name)}


EXCLUDED_DENDRITE_NAMES = normalize_excluded_dendrites(EXCLUDED_BASAL_DENDRITES)

# Mapping basal-сетей к нейронам нужен для Fig. 6a/6c.
# В metadata после парсинга RData этой информации нет, поэтому mapping вынесен явно.
BASAL_TO_NEURON = {
    "basal_01": "Neuron 1", "basal_02": "Neuron 1", "basal_03": "Neuron 1", "basal_04": "Neuron 1",
    "basal_05": "Neuron 2", "basal_06": "Neuron 2", "basal_07": "Neuron 2",
    "basal_09": "Neuron 3", "basal_10": "Neuron 3", "basal_11": "Neuron 3",
    "basal_12": "Neuron 4", "basal_13": "Neuron 4", "basal_14": "Neuron 4",
    "basal_15": "Neuron 5", "basal_16": "Neuron 5", "basal_17": "Neuron 5",
}

REFERENCE_VALUES = {
    "mean_apical_n_spines": 2845,
    "mean_apical_length": 2497.25,
    "mean_apical_branch_points": 20,
    "mean_basal_n_spines": 1074,
    "mean_basal_length": 951.85,
    "mean_basal_branch_points": 6,
    "fig6a_basal_by_neuron_r_max": 134.70,
    "fig6a_basal_by_neuron_p": 0.808,
    "fig6b_apical_vs_basal_r_max": 134.70,
    "fig6b_apical_vs_basal_p": 0.109,
    "fig6c_apical_vs_basal_excluding_neuron2_r_max": 165.96,
    "fig6c_apical_vs_basal_excluding_neuron2_p": 0.045,
    "basal_by_neuron_excluding_neuron2_p": 0.565,
}

ANALYSIS_KWARGS = {
    "root_vertex_id": ROOT_VERTEX_ID,
    "max_distance_to_edge": np.inf,
    "bin_size": "auto",
    "r_max": "circumradius",
    "n_r_values": N_R_VALUES,
    "n_simulations": N_SIMULATIONS,
    "n_jobs": N_JOBS,
    "k_correction": "geometric",
    "k_inhomogeneous": True,
    "envelope_type": "global_constant_width",
    "max_network_samples": 50_000,
    "max_integration_samples": 50_000,
    "random_state": RANDOM_STATE,
    "report_format": "html",
}

In [ ]:
network_dirs = []
for group in ("basal", "apical"):
    group_dir = DATA_ROOT / group
    network_dirs.extend(sorted(path for path in group_dir.glob(f"{group}_*") if path.is_dir()))

if EXCLUDED_DENDRITE_NAMES:
    network_dirs = [path for path in network_dirs if path.name not in EXCLUDED_DENDRITE_NAMES]
    print(f"Excluded from analysis: {sorted(EXCLUDED_DENDRITE_NAMES)}")

if NETWORK_LIMIT is not None:
    network_dirs = network_dirs[: int(NETWORK_LIMIT)]

print(f"Found {len(network_dirs)} networks")
network_dirs[:5]

## Визуализация всех базальных дендритов

На одном 3D-графике ниже показаны не исключённые из анализа basal-сети разными цветами. Подпись содержит имя дендрита и текущую привязку к нейрону из `BASAL_TO_NEURON`.

In [ ]:
basal_dirs = sorted(
    path
    for path in (DATA_ROOT / "basal").glob("basal_*")
    if path.is_dir() and path.name not in EXCLUDED_DENDRITE_NAMES
)

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")
colors = plt.cm.tab20(np.linspace(0, 1, max(len(basal_dirs), 1)))

for color, network_dir in zip(colors, basal_dirs):
    vertices = pd.read_csv(network_dir / "vertices.csv")
    edges_path = network_dir / "edges.csv"
    spines_path = network_dir / "spines.csv"
    edges = pd.read_csv(edges_path) if edges_path.exists() else None
    spines = pd.read_csv(spines_path) if spines_path.exists() else None
    label = f"{network_dir.name} ({BASAL_TO_NEURON.get(network_dir.name, 'unknown')})"

    if edges is not None and len(edges) > 0:
        coordinates = vertices.set_index("vertex_id")
        first_edge = True
        for edge in edges.itertuples(index=False):
            source = coordinates.loc[int(edge.source)]
            target = coordinates.loc[int(edge.target)]
            ax.plot(
                [source["x"], target["x"]],
                [source["y"], target["y"]],
                [source["z"], target["z"]],
                linewidth=0.8,
                alpha=0.75,
                color=color,
                label=label if first_edge else None,
            )
            first_edge = False
    else:
        ax.scatter(
            vertices["x"],
            vertices["y"],
            vertices["z"],
            s=3,
            color=color,
            alpha=0.75,
            label=label,
        )

    if spines is not None and len(spines) > 0:
        ax.scatter(
            spines["x"],
            spines["y"],
            spines["z"],
            s=1,
            color=color,
            alpha=0.25,
        )

    center = vertices[["x", "y", "z"]].mean()
    ax.text(
        center["x"],
        center["y"],
        center["z"],
        network_dir.name,
        color=color,
        fontsize=9,
        fontweight="bold",
    )

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("Все базальные дендриты")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0)

plt.tight_layout()
plt.show()


basal_dirs_by_neuron = {
    neuron: sorted(path for path in basal_dirs if BASAL_TO_NEURON.get(path.name) == neuron)
    for neuron in sorted(set(BASAL_TO_NEURON.values()))
}

for neuron, neuron_basal_dirs in basal_dirs_by_neuron.items():
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(neuron_basal_dirs), 1)))

    for color, network_dir in zip(colors, neuron_basal_dirs):
        vertices = pd.read_csv(network_dir / "vertices.csv")
        edges_path = network_dir / "edges.csv"
        spines_path = network_dir / "spines.csv"
        edges = pd.read_csv(edges_path) if edges_path.exists() else None
        spines = pd.read_csv(spines_path) if spines_path.exists() else None

        if edges is not None and len(edges) > 0:
            coordinates = vertices.set_index("vertex_id")
            first_edge = True
            for edge in edges.itertuples(index=False):
                source = coordinates.loc[int(edge.source)]
                target = coordinates.loc[int(edge.target)]
                ax.plot(
                    [source["x"], target["x"]],
                    [source["y"], target["y"]],
                    [source["z"], target["z"]],
                    linewidth=0.9,
                    alpha=0.8,
                    color=color,
                    label=network_dir.name if first_edge else None,
                )
                first_edge = False
        else:
            ax.scatter(
                vertices["x"],
                vertices["y"],
                vertices["z"],
                s=4,
                color=color,
                alpha=0.8,
                label=network_dir.name,
            )

        if spines is not None and len(spines) > 0:
            ax.scatter(
                spines["x"],
                spines["y"],
                spines["z"],
                s=1,
                color=color,
                alpha=0.25,
            )

        center = vertices[["x", "y", "z"]].mean()
        ax.text(
            center["x"],
            center["y"],
            center["z"],
            network_dir.name,
            color=color,
            fontsize=9,
            fontweight="bold",
        )

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title(f"Базальные дендриты: {neuron}")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0)

    plt.tight_layout()
    plt.show()

## Быстрая проверка структуры и аналог Table 1/Table 2

В статье Table 1/Table 2 содержат число шипиков `n`, длину сети `|L|`, плотность `n/|L|`, circumradius `R` и число branching points `#BP`. Здесь считаем тот же набор по каноническим сетям.

In [ ]:
def count_branch_points(graph, count_soma=True):
    soma_node = graph.soma_node
    return sum(
        1
        for node in graph.G.nodes()
        if graph.G.degree(node) >= 3 and (count_soma or node != soma_node)
    )


rows = []
loaded_networks = {}

for network_dir in network_dirs:
    graph, spine_points, metadata = load_standard_network(
        network_dir,
        root_vertex_id=ROOT_VERTEX_ID,
        project_spines=False,
    )
    projected, unassigned = project_spines_to_graph(graph, spine_points, max_distance_to_edge=np.inf)
    distances = np.asarray([spine.distance_to_edge for spine in projected], dtype=float)
    group = network_dir.parent.name
    neuron_id = BASAL_TO_NEURON.get(network_dir.name, network_dir.name.replace("apical_0", "Neuron "))
    branch_points = count_branch_points(graph, count_soma=COUNT_SOMA_AS_BRANCH_POINT)
    loaded_networks[network_dir.name] = {
        "group": group,
        "neuron_id": neuron_id,
        "dir": network_dir,
        "graph": graph,
        "spine_points": spine_points,
        "projected_spines": projected,
        "metadata": metadata,
    }
    rows.append({
        "network": network_dir.name,
        "group": group,
        "neuron_id": neuron_id,
        "n_spines": len(spine_points),
        "network_length": graph.total_length,
        "spine_linear_density": len(spine_points) / graph.total_length,
        "circumradius": network_circumradius(graph),
        "branch_points": branch_points,
        "root_vertex_id": graph.soma_node,
        "projection_unassigned": len(unassigned),
        "projection_distance_max": float(np.max(distances)) if len(distances) else np.nan,
        "projection_distance_median": float(np.median(distances)) if len(distances) else np.nan,
    })

structure_check = pd.DataFrame(rows).sort_values(["group", "network"]).reset_index(drop=True)
print(f"Prepared structure metrics for {len(structure_check)} networks")

In [ ]:
summary_by_group = structure_check.groupby("group")[[
    "n_spines", "network_length", "spine_linear_density", "circumradius", "branch_points"
]].mean()

In [ ]:
ROUND_COLUMNS = ["network_length", "spine_linear_density", "circumradius"]
METRIC_COLUMNS = ["n_spines", "network_length", "spine_linear_density", "circumradius", "branch_points"]


def table_with_mean_row(frame, columns):
    table = frame[columns].copy()
    mean_row = {column: "" for column in columns}
    if "network" in mean_row:
        mean_row["network"] = "mean"
    for column in METRIC_COLUMNS:
        if column in table.columns:
            mean_row[column] = round(float(table[column].mean()), 2)
    table = pd.concat([table, pd.DataFrame([mean_row])], ignore_index=True)
    for column in ROUND_COLUMNS:
        if column in table.columns:
            table[column] = pd.to_numeric(table[column], errors="coerce").round(2)
    return table


apical_table = table_with_mean_row(
    structure_check[structure_check["group"] == "apical"],
    ["network", "n_spines", "network_length", "spine_linear_density", "circumradius", "branch_points"],
)
basal_table = table_with_mean_row(
    structure_check[structure_check["group"] == "basal"],
    ["neuron_id", "network", "n_spines", "network_length", "spine_linear_density", "circumradius", "branch_points"],
)

print("Apical dendrites")
display(apical_table)
print("Basal dendrites")
display(basal_table)

## Аналог Fig. 4: сглаженная интенсивность для первого basal arbor

В статье Fig. 4 показывает kernel-smoothed estimate интенсивности как функцию расстояния до tree root для первого basal arborization of Neuron 1.

In [ ]:
def estimate_smooth_intensity_with_bootstrap_envelope(
    graph,
    spines,
    n_bootstraps=200,
    alpha=0.05,
    bandwidth=None,
    eval_step=1.0,
    max_eval_points=50_000,
    max_network_samples=50_000,
    random_state=42,
):
    if not spines:
        empty = np.array([])
        return empty, empty, empty, empty

    spine_dists = np.asarray([spine.distance_to_soma for spine in spines], dtype=float)
    n_spines = len(spine_dists)

    if bandwidth is None:
        sigma = float(np.std(spine_dists))
        if sigma < 1e-10:
            sigma = 1.0
        bandwidth = 1.06 * sigma * (n_spines ** (-0.2))
    h = max(float(bandwidth), 1e-6)

    d_max = float(spine_dists.max()) + 3.0 * h
    if max_eval_points > 0 and d_max / max(eval_step, 1e-9) > max_eval_points:
        eval_step = d_max / float(max_eval_points)
    d_grid = np.arange(0.0, d_max, eval_step)
    if len(d_grid) == 0:
        empty = np.array([])
        return d_grid, empty, empty, empty

    sample_step = max(eval_step / 2.0, 1e-6)
    _, sample_sd = graph.sample_points_on_graph(step=sample_step)
    if len(sample_sd) > max_network_samples:
        sample_step = graph.total_length / float(max_network_samples)
        _, sample_sd = graph.sample_points_on_graph(step=max(sample_step, 1e-6))

    rho_net = np.zeros(len(d_grid))
    if len(sample_sd) > 0 and graph.total_length > 0:
        sample_sd_sorted = np.sort(sample_sd)
        left = np.searchsorted(sample_sd_sorted, d_grid - h, side="left")
        right = np.searchsorted(sample_sd_sorted, d_grid + h, side="right")
        counts = right - left
        rho_net = counts.astype(float) / len(sample_sd_sorted) * graph.total_length / (2.0 * h)

    def smooth_curve(current_spine_dists):
        diff = d_grid[:, None] - current_spine_dists[None, :]
        kde = np.sum(np.exp(-0.5 * (diff / h) ** 2), axis=1) / (len(current_spine_dists) * h * np.sqrt(2.0 * np.pi))
        return np.where(rho_net > 1e-12, kde / rho_net, 0.0)

    rho_hat = smooth_curve(spine_dists)
    rng = np.random.default_rng(random_state)
    bootstrap_curves = np.empty((int(n_bootstraps), len(d_grid)), dtype=float)
    for index in range(int(n_bootstraps)):
        resampled = rng.choice(spine_dists, size=n_spines, replace=True)
        bootstrap_curves[index] = smooth_curve(resampled)

    rho_lo = np.quantile(bootstrap_curves, alpha / 2.0, axis=0)
    rho_hi = np.quantile(bootstrap_curves, 1.0 - alpha / 2.0, axis=0)
    return d_grid, rho_hat, rho_lo, rho_hi


example_name = "basal_01"
example = loaded_networks[example_name]
smooth_d, smooth_lambda, rho_lo, rho_hi = estimate_smooth_intensity_with_bootstrap_envelope(
    example["graph"],
    example["projected_spines"],
    n_bootstraps=INTENSITY_ENVELOPE_BOOTSTRAPS,
    alpha=INTENSITY_ENVELOPE_ALPHA,
    random_state=RANDOM_STATE,
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(smooth_d, rho_lo, rho_hi, color="tab:red", alpha=0.18, label=r"$\rho_{lo}(d)$ - $\rho_{hi}(d)$")
ax.plot(smooth_d, smooth_lambda, color="tab:red", label=r"$\hat{\rho}(d)$")
ax.set_title(f"Kernel-smoothed intensity: {example_name}")
ax.set_xlabel("Network distance to root")
ax.set_ylabel("Estimated intensity")
ax.legend()
ax.grid(alpha=0.25)
fig

## Полный сетевой анализ каждой сети

Эта ячейка запускает весь reference-like pipeline: CDF-test, log-quadratic inhomogeneous Poisson model, geometrically corrected inhomogeneous network K-function и global constant-width envelope.

In [ ]:
ANALYSIS_CACHE_VERSION = 1


def _cache_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): _cache_jsonable(value[key]) for key in sorted(value)}
    if isinstance(value, (list, tuple)):
        return [_cache_jsonable(item) for item in value]
    if isinstance(value, set):
        return sorted(_cache_jsonable(item) for item in value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return "inf" if np.isinf(value) else float(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    return value


def _network_input_signature(network_dir):
    entries = []
    for file_name in ("vertices.csv", "edges.csv", "spines.csv", "matrix.npy", "metadata.json"):
        file_path = Path(network_dir) / file_name
        if file_path.exists():
            stat = file_path.stat()
            entries.append({
                "name": file_name,
                "size": int(stat.st_size),
                "mtime_ns": int(stat.st_mtime_ns),
            })
        else:
            entries.append({"name": file_name, "missing": True})
    return entries


def analysis_cache_key(network_dir):
    payload = {
        "cache_version": ANALYSIS_CACHE_VERSION,
        "network": Path(network_dir).name,
        "group": Path(network_dir).parent.name,
        "inputs": _network_input_signature(network_dir),
        "analysis_kwargs": _cache_jsonable(ANALYSIS_KWARGS),
    }
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:16]


def analysis_cache_path(network_dir):
    key = analysis_cache_key(network_dir)
    return ANALYSIS_CACHE_DIR / f"{Path(network_dir).parent.name}__{Path(network_dir).name}__{key}.pkl"


def load_analysis_from_cache(network_dir):
    if not USE_ANALYSIS_CACHE or FORCE_RECOMPUTE_ANALYSIS:
        return None
    cache_path = analysis_cache_path(network_dir)
    if not cache_path.exists():
        return None
    try:
        with cache_path.open("rb") as handle:
            payload = pickle.load(handle)
        result = payload["result"] if isinstance(payload, dict) and "result" in payload else payload
        result["cache_path"] = str(cache_path)
        return result
    except Exception as exc:
        print(f"[cache] failed to load {cache_path.name}: {type(exc).__name__}: {exc}")
        return None


def save_analysis_to_cache(network_dir, result):
    if not USE_ANALYSIS_CACHE:
        return ""
    ANALYSIS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_path = analysis_cache_path(network_dir)
    payload = {
        "cache_version": ANALYSIS_CACHE_VERSION,
        "network": Path(network_dir).name,
        "group": Path(network_dir).parent.name,
        "analysis_kwargs": _cache_jsonable(ANALYSIS_KWARGS),
        "result": result,
    }
    try:
        with cache_path.open("wb") as handle:
            pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
        result["cache_path"] = str(cache_path)
        return str(cache_path)
    except Exception as exc:
        print(f"[cache] failed to save {cache_path.name}: {type(exc).__name__}: {exc}")
        return ""


def analysis_summary_row(network_dir, result, source):
    k_result = result.get("k_result")
    poisson_result = result.get("poisson_result")
    graph = result["graph"]
    return {
        "network": network_dir.name,
        "group": network_dir.parent.name,
        "source": source,
        "n_spines": len(result["projected_spines"]),
        "network_length": graph.total_length,
        "spine_linear_density": len(result["projected_spines"]) / graph.total_length,
        "circumradius": network_circumradius(graph),
        "cdf_p_value": result["intensity_cdf_test"].get("p_value"),
        "k_p_value": None if k_result is None else k_result.p_value,
        "poisson_log_likelihood": None if poisson_result is None else poisson_result.log_likelihood,
        "report_path": result.get("report_path", ""),
        "cache_path": result.get("cache_path", ""),
    }


analysis_rows = []
full_results = {}
missing_analysis = []

for network_dir in network_dirs:
    result = load_analysis_from_cache(network_dir)
    if result is not None:
        print(f"[cache] loaded {network_dir.parent.name}/{network_dir.name}")
        full_results[network_dir.name] = result
        analysis_rows.append(analysis_summary_row(network_dir, result, source="cache"))
        continue

    if not RUN_FULL_ANALYSIS:
        missing_analysis.append(f"{network_dir.parent.name}/{network_dir.name}")
        continue

    print(f"[Anton-Sanchez] full analysis start {network_dir.parent.name}/{network_dir.name}", flush=True)
    result = run_analysis(
        standard_network_dir=network_dir,
        output_dir=str(OUTPUT_DIR / network_dir.parent.name / network_dir.name),
        dendrite_type=network_dir.parent.name,
        timing_label=f"{network_dir.parent.name}/{network_dir.name}",
        **ANALYSIS_KWARGS,
    )
    cache_path = save_analysis_to_cache(network_dir, result)
    if cache_path:
        print(f"[cache] saved {network_dir.parent.name}/{network_dir.name} -> {cache_path}")
    full_results[network_dir.name] = result
    analysis_rows.append(analysis_summary_row(network_dir, result, source="computed"))

if missing_analysis:
    print(
        "RUN_FULL_ANALYSIS=False: нет кэша для "
        f"{len(missing_analysis)} сетей. Включи RUN_FULL_ANALYSIS=True, чтобы досчитать: "
        + ", ".join(missing_analysis[:5])
        + ("..." if len(missing_analysis) > 5 else "")
    )

analysis_summary = pd.DataFrame(analysis_rows)
analysis_summary


## Аналог Fig. 5: KLI для basal arborization Neuron 1

Эта ячейка строит три панели для одного и того же набора шипиков basal-дендритов `Neuron 1`: 3D network KLI, 2D-projected network KLI и обычную 3D spatial K-функцию без учёта дендритной сети. Для network-панелей используется та же log-quadratic inhomogeneous Poisson intensity по расстоянию до сомы и global constant-width Monte Carlo envelope.


In [ ]:
FIG5_TARGET_NEURON = "Neuron 1"
if "loaded_networks" not in globals():
    loaded_networks = {}

FIG5_TARGET_NETWORKS = [
    name
    for name, neuron in BASAL_TO_NEURON.items()
    if neuron == FIG5_TARGET_NEURON and name not in EXCLUDED_DENDRITE_NAMES
]


def ensure_loaded_basal_network(network_name):
    global loaded_networks
    if network_name in loaded_networks:
        return loaded_networks[network_name]
    network_dir = DATA_ROOT / "basal" / network_name
    graph, spine_points, metadata = load_standard_network(
        network_dir,
        root_vertex_id=ROOT_VERTEX_ID,
        project_spines=False,
    )
    projected, unassigned = project_spines_to_graph(graph, spine_points, max_distance_to_edge=np.inf)
    item = {
        "group": "basal",
        "neuron_id": BASAL_TO_NEURON.get(network_name, "unknown"),
        "dir": network_dir,
        "graph": graph,
        "spine_points": spine_points,
        "projected_spines": projected,
        "metadata": metadata,
        "projection_unassigned": unassigned,
    }
    loaded_networks[network_name] = item
    return item


def _project_xy(point):
    point = np.asarray(point, dtype=float).copy()
    point[2] = 0.0
    return point


def _network_root_node(graph):
    if graph.soma_node in graph.G:
        return graph.soma_node
    if ROOT_VERTEX_ID in graph.G:
        return ROOT_VERTEX_ID
    return next(iter(graph.G.nodes()))


def combine_basal_networks_for_neuron(network_names, project_to_xy=False):
    items = [(name, ensure_loaded_basal_network(name)) for name in network_names]
    if not items:
        raise ValueError(f"No basal networks found for {FIG5_TARGET_NEURON}.")

    combined = network_module.DendriticGraph()
    soma_node = -1
    root_positions = []
    for _, item in items:
        graph = item["graph"]
        root = _network_root_node(graph)
        root_pos = graph.node_position(root)
        root_positions.append(_project_xy(root_pos) if project_to_xy else np.asarray(root_pos, dtype=float))

    soma_pos = np.mean(np.vstack(root_positions), axis=0) if root_positions else np.zeros(3)
    combined.G.add_node(
        soma_node,
        pos=soma_pos,
        node_type="soma",
        dendrite_id=FIG5_TARGET_NEURON,
        dendrite_type="basal",
    )
    combined.soma_node = soma_node

    mappings = {}
    next_node = 0
    for network_name, item in items:
        graph = item["graph"]
        mapping = {}
        for node, data in graph.G.nodes(data=True):
            new_node = next_node
            next_node += 1
            mapping[node] = new_node
            attrs = dict(data)
            pos = graph.node_position(node)
            attrs["pos"] = _project_xy(pos) if project_to_xy else np.asarray(pos, dtype=float)
            attrs["source_network"] = network_name
            attrs["dendrite_id"] = network_name
            attrs["dendrite_type"] = "basal"
            combined.G.add_node(new_node, **attrs)

        for u, v, data in graph.G.edges(data=True):
            u_new = mapping[u]
            v_new = mapping[v]
            attrs = dict(data)
            if project_to_xy:
                length = float(np.linalg.norm(combined.node_position(u_new) - combined.node_position(v_new)))
            else:
                length = float(data.get("length", np.linalg.norm(combined.node_position(u_new) - combined.node_position(v_new))))
            attrs["length"] = length
            attrs["source_network"] = network_name
            combined.G.add_edge(u_new, v_new, **attrs)

        root = _network_root_node(graph)
        combined.G.add_edge(
            soma_node,
            mapping[root],
            length=0.0,
            source_network=network_name,
            synthetic_soma_edge=True,
        )
        mappings[network_name] = mapping

    combined._soma_distances = None
    soma_distances = combined.soma_distances(recompute=True)
    combined_spines = []
    for network_name, item in items:
        mapping = mappings[network_name]
        for spine in item["projected_spines"]:
            u = mapping[spine.edge_source]
            v = mapping[spine.edge_target]
            edge_length = float(combined.G[u][v].get("length", 0.0))
            projected_point = combined.node_position(u) + spine.edge_position * (combined.node_position(v) - combined.node_position(u))
            original_point = _project_xy(spine.original_point) if project_to_xy else np.asarray(spine.original_point, dtype=float)
            distance_to_soma = min(
                soma_distances.get(u, np.inf) + spine.edge_position * edge_length,
                soma_distances.get(v, np.inf) + (1.0 - spine.edge_position) * edge_length,
            )
            combined_spines.append(network_module.ProjectedSpine(
                spine_id=f"{network_name}:{spine.spine_id}",
                original_point=original_point,
                projected_point=projected_point,
                edge_id=f"{network_name}:{spine.edge_id}",
                edge_source=u,
                edge_target=v,
                edge_position=float(spine.edge_position),
                distance_to_edge=float(spine.distance_to_edge),
                distance_to_soma=float(distance_to_soma),
            ))

    return combined, combined_spines


def compute_network_kli_envelope(graph, spines, r_values, label):
    poisson = fit_inhomogeneous_poisson(
        graph,
        spines,
        max_integration_samples=ANALYSIS_KWARGS.get("max_integration_samples", 50_000),
        verbose_timing=True,
        timing_label=label,
    )
    result = compute_simulation_envelopes(
        graph,
        spines,
        r_values,
        n_simulations=N_SIMULATIONS,
        alpha=0.05,
        n_jobs=N_JOBS,
        intensity_model=poisson,
        correction="geometric",
        envelope_type="global_constant_width",
        require_inhomogeneous=True,
        random_state=RANDOM_STATE,
        timing_label=label,
    )
    return {
        "d": result.r_values,
        "K_observed": result.k_observed,
        "K_theoretical": result.k_expected,
        "K_lower": result.k_lower,
        "K_upper": result.k_upper,
        "p_value": result.p_value,
        "raw_result": result,
        "poisson_model": poisson,
    }


def euclidean_k_3d(points, r_values, volume, chunk_size=512):
    points = np.asarray(points, dtype=float)
    r_values = np.asarray(r_values, dtype=float)
    n_points = len(points)
    if n_points < 2 or volume <= 0:
        return np.zeros_like(r_values)

    counts = np.zeros(len(r_values), dtype=float)
    for start in range(0, n_points, chunk_size):
        stop = min(start + chunk_size, n_points)
        diff = points[start:stop, None, :] - points[None, :, :]
        distances = np.sqrt(np.sum(diff * diff, axis=2))
        distances[np.arange(stop - start), np.arange(start, stop)] = np.inf
        distances_sorted = np.sort(distances.ravel())
        counts += np.searchsorted(distances_sorted, r_values, side="right")
    return volume * counts / (n_points * (n_points - 1))


def compute_spatial_3d_k_envelope(points, r_values, n_simulations=19, alpha=0.05, random_state=42):
    points = np.asarray(points, dtype=float)
    bounds_min = points.min(axis=0)
    bounds_max = points.max(axis=0)
    spans = np.maximum(bounds_max - bounds_min, 1e-9)
    volume = float(np.prod(spans))
    observed = euclidean_k_3d(points, r_values, volume)
    theoretical = (4.0 / 3.0) * np.pi * np.asarray(r_values, dtype=float) ** 3

    rng = np.random.default_rng(random_state)
    n_simulations = int(max(0, n_simulations))
    if n_simulations == 0:
        return {
            "d": np.asarray(r_values, dtype=float),
            "K_observed": observed,
            "K_theoretical": theoretical,
            "K_lower": None,
            "K_upper": None,
            "p_value": np.nan,
        }

    simulated_curves = []
    for _ in range(n_simulations):
        simulated_points = rng.uniform(bounds_min, bounds_max, size=points.shape)
        simulated_curves.append(euclidean_k_3d(simulated_points, r_values, volume))
    simulated_curves = np.asarray(simulated_curves, dtype=float)
    sim_devs = np.max(np.abs(simulated_curves - theoretical[None, :]), axis=1)
    obs_dev = float(np.max(np.abs(observed - theoretical)))
    envelope_rank = int(np.ceil((1.0 - alpha) * (n_simulations + 1)))
    envelope_rank = int(np.clip(envelope_rank, 1, n_simulations))
    w_max = float(np.sort(sim_devs)[envelope_rank - 1])

    return {
        "d": np.asarray(r_values, dtype=float),
        "K_observed": observed,
        "K_theoretical": theoretical,
        "K_lower": theoretical - w_max,
        "K_upper": theoretical + w_max,
        "p_value": float((1 + np.sum(sim_devs >= obs_dev)) / (1 + n_simulations)),
    }


def plot_k_panel(ax, result, title, ylabel):
    d = result["d"]
    lower = result.get("K_lower")
    upper = result.get("K_upper")
    if lower is not None and upper is not None:
        ax.fill_between(d, lower, upper, color="0.55", alpha=0.22, label="Monte Carlo envelope")
        ax.plot(d, lower, color="0.45", linewidth=0.9)
        ax.plot(d, upper, color="0.45", linewidth=0.9)
    ax.plot(d, result["K_observed"], color="tab:blue", linewidth=2.0, label="observed")
    ax.plot(d, result["K_theoretical"], color="black", linestyle="--", linewidth=1.4, label="theoretical Poisson")
    p_value = result.get("p_value")
    p_text = "n/a" if p_value is None or not np.isfinite(p_value) else f"{p_value:.3f}"
    ax.set_title(f"{title}\np = {p_text}")
    ax.set_xlabel("d")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)


fig5_results = {}
if FIG5_TARGET_NETWORKS:
    fig5_graph_3d, fig5_spines_3d = combine_basal_networks_for_neuron(FIG5_TARGET_NETWORKS, project_to_xy=False)
    fig5_graph_2d, fig5_spines_2d = combine_basal_networks_for_neuron(FIG5_TARGET_NETWORKS, project_to_xy=True)
    fig5_r_max = min(network_circumradius(fig5_graph_3d), network_circumradius(fig5_graph_2d))
    fig5_r_values = np.linspace(0.0, fig5_r_max, N_R_VALUES + 1)[1:]

    fig5_results["3d_network"] = compute_network_kli_envelope(
        fig5_graph_3d,
        fig5_spines_3d,
        fig5_r_values,
        f"Fig5 {FIG5_TARGET_NEURON} 3D network",
    )
    fig5_results["2d_projected_network"] = compute_network_kli_envelope(
        fig5_graph_2d,
        fig5_spines_2d,
        fig5_r_values,
        f"Fig5 {FIG5_TARGET_NEURON} 2D projected network",
    )
    fig5_points_3d = np.vstack([spine.original_point for spine in fig5_spines_3d])
    fig5_results["3d_spatial"] = compute_spatial_3d_k_envelope(
        fig5_points_3d,
        fig5_r_values,
        n_simulations=N_SIMULATIONS,
        alpha=0.05,
        random_state=RANDOM_STATE,
    )

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    plot_k_panel(axes[0], fig5_results["3d_network"], "Fig. 5a: 3D network KLI", "KLI(d)")
    plot_k_panel(axes[1], fig5_results["2d_projected_network"], "Fig. 5b: 2D projected network KLI", "KLI(d)")
    plot_k_panel(axes[2], fig5_results["3d_spatial"], "Fig. 5c: ordinary 3D spatial K", "K(d)")
    fig.suptitle(f"Basal arbors of {FIG5_TARGET_NEURON}: {', '.join(FIG5_TARGET_NETWORKS)}", y=1.03)
    fig.tight_layout()
    display(fig)
else:
    print(f"No basal dendrites found for {FIG5_TARGET_NEURON} after exclusions.")

fig5_results


## Аналоги Fig. 6: индивидуальные KLI-кривые и studentized permutation test

Для каждой панели KLI-кривые приводятся к общему диапазону расстояний: `d_max = (1 - CIRCUMRADIUS_MARGIN) * min(circumradius)` для включённых dendritic patterns. Для Fig. 6c дополнительно исключается `Neuron 2`, а целевой диапазон берётся как `[0, 165.96]`, если он не превышает минимальный circumradius оставшихся сетей.


In [ ]:
KLI_PATTERN_CACHE = {}
if "loaded_networks" not in globals():
    loaded_networks = {}


def ensure_network_dirs_for_figures():
    global network_dirs
    if "network_dirs" in globals():
        return network_dirs

    network_dirs = []
    for group in ("basal", "apical"):
        group_dir = DATA_ROOT / group
        network_dirs.extend(sorted(path for path in group_dir.glob(f"{group}_*") if path.is_dir()))

    if EXCLUDED_DENDRITE_NAMES:
        network_dirs = [path for path in network_dirs if path.name not in EXCLUDED_DENDRITE_NAMES]
    if NETWORK_LIMIT is not None:
        network_dirs = network_dirs[: int(NETWORK_LIMIT)]
    return network_dirs


def ensure_loaded_network_for_figures(network_name, group=None):
    global loaded_networks
    if network_name in loaded_networks:
        return loaded_networks[network_name]

    if group is None:
        group = "basal" if network_name.startswith("basal_") else "apical"
    network_dir = DATA_ROOT / group / network_name
    graph, spine_points, metadata = load_standard_network(
        network_dir,
        root_vertex_id=ROOT_VERTEX_ID,
        project_spines=False,
    )
    projected, unassigned = project_spines_to_graph(graph, spine_points, max_distance_to_edge=np.inf)
    item = {
        "group": group,
        "neuron_id": BASAL_TO_NEURON.get(network_name, network_name.replace("apical_0", "Neuron ")),
        "dir": network_dir,
        "graph": graph,
        "spine_points": spine_points,
        "projected_spines": projected,
        "metadata": metadata,
        "projection_unassigned": unassigned,
    }
    loaded_networks[network_name] = item
    return item


def dendrite_neuron_id(network_name):
    if network_name in BASAL_TO_NEURON:
        return BASAL_TO_NEURON[network_name]
    if network_name.startswith("apical_"):
        suffix = network_name.split("_", 1)[1]
        try:
            return f"Neuron {int(suffix)}"
        except ValueError:
            return network_name
    return ensure_loaded_network_for_figures(network_name).get("neuron_id", network_name)


def dendrite_type_label(network_name):
    group = ensure_loaded_network_for_figures(network_name)["group"]
    return "Basal" if group == "basal" else "Apical"


def common_distance_grid(network_names, circumradius_margin=0.02, n_grid=300, forced_d_max=None):
    radii = {name: network_circumradius(ensure_loaded_network_for_figures(name)["graph"]) for name in network_names}
    positive_radii = [radius for radius in radii.values() if radius > 0]
    if not network_names or not positive_radii:
        return np.array([], dtype=float), np.nan, radii
    min_radius = float(min(positive_radii))
    if forced_d_max is None:
        d_max = (1.0 - float(circumradius_margin)) * min_radius
    else:
        d_max = float(forced_d_max)
        if d_max > min_radius:
            print(
                f"Requested d_max={d_max:.2f} exceeds min circumradius={min_radius:.2f}; "
                "clipping to avoid extrapolation."
            )
            d_max = min_radius
    d_common = np.linspace(0.0, d_max, int(n_grid))
    return d_common, d_max, radii


def compute_kli_pattern(network_name, d_values):
    d_values = np.asarray(d_values, dtype=float)
    cache_key = (network_name, len(d_values), round(float(d_values[0]), 6), round(float(d_values[-1]), 6))
    if cache_key in KLI_PATTERN_CACHE:
        return KLI_PATTERN_CACHE[cache_key]

    item = ensure_loaded_network_for_figures(network_name)
    cached_analysis = globals().get("full_results", {}).get(network_name)
    cached_k = None if cached_analysis is None else cached_analysis.get("k_result")
    if cached_k is not None and len(cached_k.r_values) > 0:
        source_d = np.asarray(cached_k.r_values, dtype=float)
        source_k = np.asarray(cached_k.k_observed, dtype=float)
        if source_d[0] > 0:
            source_d = np.concatenate(([0.0], source_d))
            source_k = np.concatenate(([0.0], source_k))
        if source_d[0] <= d_values[0] + 1e-9 and source_d[-1] >= d_values[-1] - 1e-9:
            pattern = {
                "network": network_name,
                "d": d_values,
                "K_observed": np.interp(d_values, source_d, source_k),
                "circumradius": network_circumradius(item["graph"]),
                "neuron_id": dendrite_neuron_id(network_name),
                "dendrite_type": dendrite_type_label(network_name),
                "source": "full_results",
            }
            KLI_PATTERN_CACHE[cache_key] = pattern
            return pattern

    poisson = fit_inhomogeneous_poisson(
        item["graph"],
        item["projected_spines"],
        max_integration_samples=ANALYSIS_KWARGS.get("max_integration_samples", 50_000),
        verbose_timing=False,
    )
    kli = ripley_k_network(
        item["graph"],
        item["projected_spines"],
        d_values,
        intensity=poisson.fitted_intensity,
        correction="geometric",
        n_jobs=N_JOBS,
    )
    pattern = {
        "network": network_name,
        "d": kli.r_values,
        "K_observed": kli.k_observed,
        "circumradius": network_circumradius(item["graph"]),
        "neuron_id": dendrite_neuron_id(network_name),
        "dendrite_type": dendrite_type_label(network_name),
        "source": "computed",
    }
    KLI_PATTERN_CACHE[cache_key] = pattern
    return pattern


def compute_kli_patterns(network_names, d_values):
    if len(d_values) == 0:
        return []
    return [compute_kli_pattern(name, d_values) for name in network_names]


def interpolate_patterns_to_common_grid(patterns, d_common):
    curves = []
    labels = []
    names = []
    d_common = np.asarray(d_common, dtype=float)
    for pattern in patterns:
        d = np.asarray(pattern["d"], dtype=float)
        k = np.asarray(pattern["K_observed"], dtype=float)
        if d_common[0] < d[0] - 1e-9 or d_common[-1] > d[-1] + 1e-9:
            raise ValueError(
                f"Pattern {pattern['network']} does not cover d_range "
                f"[{d_common[0]:.2f}, {d_common[-1]:.2f}]."
            )
        curves.append(np.interp(d_common, d, k))
        names.append(pattern["network"])
    return np.asarray(curves, dtype=float), names


def studentized_permutation_test(patterns, group, d_range, n_permutations=1000, random_state=42, n_grid=300):
    selected = [pattern for pattern in patterns if group in pattern]
    group_names = sorted({pattern[group] for pattern in selected})
    if len(group_names) < 2 or len(selected) < 2:
        return {
            "p_value": np.nan,
            "observed_statistic": np.nan,
            "permutation_statistics": np.array([]),
            "group_count": len(group_names),
            "pattern_count": len(selected),
            "d_max": float(d_range[1]),
            "reason": "Need at least two groups and two patterns.",
        }

    d_common = np.linspace(float(d_range[0]), float(d_range[1]), int(n_grid))
    all_curves, _ = interpolate_patterns_to_common_grid(selected, d_common)
    labels = np.asarray([pattern[group] for pattern in selected], dtype=object)
    rng = np.random.default_rng(random_state)

    def statistic(current_labels):
        group_means = []
        group_vars = []
        weights = []
        for group_name in group_names:
            group_curves = all_curves[current_labels == group_name]
            group_means.append(group_curves.mean(axis=0))
            group_vars.append(group_curves.var(axis=0, ddof=1) if len(group_curves) > 1 else np.zeros(len(d_common)))
            weights.append(len(group_curves))
        group_means = np.asarray(group_means, dtype=float)
        group_vars = np.asarray(group_vars, dtype=float)
        weights = np.asarray(weights, dtype=float)
        grand_mean = np.average(group_means, axis=0, weights=weights)
        pooled_var = np.nanmean(group_vars, axis=0)
        eps = np.nanmedian(pooled_var[pooled_var > 0]) * 1e-6 if np.any(pooled_var > 0) else 1e-12
        standardized = ((group_means - grand_mean) ** 2) / (pooled_var + eps)
        return float(np.trapz(np.sum(standardized, axis=0), d_common))

    observed = statistic(labels)
    permutation_statistics = np.empty(int(n_permutations), dtype=float)
    for index in range(int(n_permutations)):
        permutation_statistics[index] = statistic(rng.permutation(labels))
    p_value = float((1 + np.sum(permutation_statistics >= observed)) / (1 + len(permutation_statistics)))
    return {
        "p_value": p_value,
        "observed_statistic": observed,
        "permutation_statistics": permutation_statistics,
        "group_count": len(group_names),
        "pattern_count": len(selected),
        "d_max": float(d_range[1]),
    }


def _format_p_value(value):
    return "n/a" if value is None or not np.isfinite(value) else f"{value:.3f}"


def plot_individual_kli_patterns(ax, patterns, group, d_range, title, test_result=None, label_map=None, colors=None):
    if not patterns:
        ax.set_axis_off()
        ax.set_title(f"{title}\nno patterns")
        return

    d_common = np.linspace(float(d_range[0]), float(d_range[1]), FIG6_COMMON_GRID_N)
    group_names = sorted({pattern[group] for pattern in patterns})
    if colors is None:
        palette = plt.cm.tab10(np.linspace(0, 1, max(len(group_names), 1)))
        colors = {group_name: palette[index] for index, group_name in enumerate(group_names)}
    plotted_labels = set()

    for pattern in patterns:
        curve = np.interp(d_common, pattern["d"], pattern["K_observed"])
        group_name = pattern[group]
        label = label_map.get(group_name, group_name) if label_map else group_name
        ax.plot(
            d_common,
            curve,
            color=colors[group_name],
            alpha=0.58,
            linewidth=1.2,
            label=label if group_name not in plotted_labels else None,
        )
        plotted_labels.add(group_name)

    if test_result is None:
        subtitle = f"g = {len(group_names)}"
    else:
        subtitle = f"g = {test_result['group_count']}, p = {_format_p_value(test_result['p_value'])}, d_max = {test_result['d_max']:.2f}"
    ax.set_title(f"{title}\n{subtitle}")
    ax.set_xlabel("d")
    ax.set_ylabel("KLI(d)")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)


In [ ]:
comparison_results = {}
fig6_patterns = {}

if RUN_GROUP_COMPARISON:
    for network_dir in ensure_network_dirs_for_figures():
        ensure_loaded_network_for_figures(network_dir.name, group=network_dir.parent.name)

    basal_names = sorted(
        name
        for name, item in loaded_networks.items()
        if item["group"] == "basal" and name in BASAL_TO_NEURON
    )
    apical_names = sorted(name for name, item in loaded_networks.items() if item["group"] == "apical")

    # Fig. 6a: individual basal dendrites grouped by neuron.
    d6a, d6a_max, _ = common_distance_grid(
        basal_names,
        circumradius_margin=CIRCUMRADIUS_MARGIN,
        n_grid=FIG6_COMMON_GRID_N,
    )
    fig6_patterns["fig6a_basal_by_neuron"] = compute_kli_patterns(basal_names, d6a)
    comparison_results["fig6a_basal_by_neuron"] = studentized_permutation_test(
        fig6_patterns["fig6a_basal_by_neuron"],
        group="neuron_id",
        d_range=(0.0, d6a_max),
        n_permutations=N_PERMUTATIONS,
        random_state=RANDOM_STATE,
        n_grid=FIG6_COMMON_GRID_N,
    )

    # Additional apical analogue requested here: individual apical dendrites grouped by neuron.
    d6a_apical, d6a_apical_max, _ = common_distance_grid(
        apical_names,
        circumradius_margin=CIRCUMRADIUS_MARGIN,
        n_grid=FIG6_COMMON_GRID_N,
    )
    fig6_patterns["fig6a_apical_by_neuron"] = compute_kli_patterns(apical_names, d6a_apical)
    comparison_results["fig6a_apical_by_neuron"] = studentized_permutation_test(
        fig6_patterns["fig6a_apical_by_neuron"],
        group="neuron_id",
        d_range=(0.0, d6a_apical_max),
        n_permutations=N_PERMUTATIONS,
        random_state=RANDOM_STATE,
        n_grid=FIG6_COMMON_GRID_N,
    )

    # Fig. 6b: individual basal and apical dendrites grouped by dendrite type.
    fig6b_names = basal_names + apical_names
    d6b, d6b_max, _ = common_distance_grid(
        fig6b_names,
        circumradius_margin=CIRCUMRADIUS_MARGIN,
        n_grid=FIG6_COMMON_GRID_N,
    )
    fig6_patterns["fig6b_basal_vs_apical"] = compute_kli_patterns(fig6b_names, d6b)
    comparison_results["fig6b_basal_vs_apical"] = studentized_permutation_test(
        fig6_patterns["fig6b_basal_vs_apical"],
        group="dendrite_type",
        d_range=(0.0, d6b_max),
        n_permutations=N_PERMUTATIONS,
        random_state=RANDOM_STATE,
        n_grid=FIG6_COMMON_GRID_N,
    )

    # Fig. 6c: repeat basal-vs-apical after excluding Neuron 2; target article range is [0, 165.96].
    fig6c_names = [name for name in fig6b_names if dendrite_neuron_id(name) != "Neuron 2"]
    d6c, d6c_max, _ = common_distance_grid(
        fig6c_names,
        circumradius_margin=CIRCUMRADIUS_MARGIN,
        n_grid=FIG6_COMMON_GRID_N,
        forced_d_max=FIG6C_R_MAX,
    )
    fig6_patterns["fig6c_basal_vs_apical_without_neuron2"] = compute_kli_patterns(fig6c_names, d6c)
    comparison_results["fig6c_basal_vs_apical_without_neuron2"] = studentized_permutation_test(
        fig6_patterns["fig6c_basal_vs_apical_without_neuron2"],
        group="dendrite_type",
        d_range=(0.0, d6c_max),
        n_permutations=N_PERMUTATIONS,
        random_state=RANDOM_STATE,
        n_grid=FIG6_COMMON_GRID_N,
    )

    neuron_groups = sorted({dendrite_neuron_id(name) for name in basal_names + apical_names})
    neuron_palette = plt.cm.tab10(np.linspace(0, 1, max(len(neuron_groups), 1)))
    neuron_colors = {neuron: neuron_palette[index] for index, neuron in enumerate(neuron_groups)}
    type_colors = {"Basal": "tab:blue", "Apical": "tab:orange"}
    type_labels = {"Basal": "Basal dendrites", "Apical": "Apical dendrites"}

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    plot_individual_kli_patterns(
        axes[0, 0],
        fig6_patterns["fig6a_basal_by_neuron"],
        group="neuron_id",
        d_range=(0.0, d6a_max),
        title="Fig. 6a: basal dendrites grouped by neuron",
        test_result=comparison_results["fig6a_basal_by_neuron"],
        colors=neuron_colors,
    )
    plot_individual_kli_patterns(
        axes[0, 1],
        fig6_patterns["fig6a_apical_by_neuron"],
        group="neuron_id",
        d_range=(0.0, d6a_apical_max),
        title="Apical analogue: dendrites grouped by neuron",
        test_result=comparison_results["fig6a_apical_by_neuron"],
        colors=neuron_colors,
    )
    plot_individual_kli_patterns(
        axes[1, 0],
        fig6_patterns["fig6b_basal_vs_apical"],
        group="dendrite_type",
        d_range=(0.0, d6b_max),
        title="Fig. 6b: basal vs apical",
        test_result=comparison_results["fig6b_basal_vs_apical"],
        label_map=type_labels,
        colors=type_colors,
    )
    plot_individual_kli_patterns(
        axes[1, 1],
        fig6_patterns["fig6c_basal_vs_apical_without_neuron2"],
        group="dendrite_type",
        d_range=(0.0, d6c_max),
        title="Fig. 6c: basal vs apical without Neuron 2",
        test_result=comparison_results["fig6c_basal_vs_apical_without_neuron2"],
        label_map=type_labels,
        colors=type_colors,
    )
    fig.tight_layout()
    display(fig)
else:
    print("RUN_GROUP_COMPARISON=False: включи True для studentized permutation comparisons.")

REFERENCE_P_VALUES = {
    "fig6a_basal_by_neuron": REFERENCE_VALUES["fig6a_basal_by_neuron_p"],
    "fig6b_basal_vs_apical": REFERENCE_VALUES["fig6b_apical_vs_basal_p"],
    "fig6c_basal_vs_apical_without_neuron2": REFERENCE_VALUES["fig6c_apical_vs_basal_excluding_neuron2_p"],
}

pd.DataFrame([
    {
        "comparison": name,
        "n_patterns": result.get("pattern_count"),
        "g": result.get("group_count"),
        "d_max": result.get("d_max"),
        "our_p_value": result.get("p_value"),
        "reference_p_value": REFERENCE_P_VALUES.get(name),
        "observed_statistic": result.get("observed_statistic"),
        "note": result.get("reason", ""),
    }
    for name, result in comparison_results.items()
])
